In [1]:
pip install langchain, langchain-openai, langchain-mcp-adapters, mcp, python-dotenv

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'langchain,': Expected semicolon (after name with no version specifier) or end
    langchain,
             ^


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing. Check your .env file.")

print("OpenAI API key is loaded.")

OpenAI API key is loaded.


In [3]:
key= os.getenv("OPENAI_API_KEY")


# Step 2: Connect to  MCP server


In [4]:
%pip install mcp-server-fetch


Note: you may need to restart the kernel to use updated packages.


1. World Bank Data360 – meine Empfehlung
- Daten: BIP, Wachstum, Armut, Arbeitsmarkt und weitere Entwicklungsindikatoren.
- Funktionen: Indikatoren suchen, Definitionen lesen und Zeitreihen abrufen. Er bietet laut Dokumentation auch MCP-Resources an – hilfreich für deinen späteren Lab-Schritt.
- Beispielfrage: „Vergleiche die Entwicklung des realen BIP pro Kopf in Deutschland und Polen.“
- Zugang: Die Weltbank dokumentiert einen öffentlichen Remote-Endpunkt; die Verbindung müssten wir noch testen.
- URL: https://maimcpext.worldbank.org/ext/data360/mcp
  Offizielle Dokumentation

In [8]:
%pip install langchain-mcp-adapters mcp



  Using cached mcp-1.30.0-py3-none-any.whl.metadata (9.3 kB)
Using cached mcp-1.30.0-py3-none-any.whl (234 kB)
  Attempting uninstall: mcp
    Found existing installation: mcp 2.3.0
    Uninstalling mcp-2.3.0:
      Successfully uninstalled mcp-2.3.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp-server-fetch 2026.10.24 requires mcp<3,>=2.3, but you have mcp 1.30.0 which is incompatible.


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

#This client manages connections to MCP servers. Despite its name, it works with just one server.
#You don’t need additional imports from mcp here: the adapter handles the underlying connection.

In [ ]:
client = MultiServerMCPClient({
    "data360": {
        "url": "https://maimcpext.worldbank.org/ext/data360/mcp",
        "transport": "http",
    }
}) # configuring the connection!

In [14]:
async with client.session("data360") as session:
    # List available tools
    tools_result = await session.list_tools()

    print("=== TOOLS ===")
    for tool in tools_result.tools:
        print(f"\nName: {tool.name}")
        print(f"Description: {tool.description}")

    # List available resources
    resources_result = await session.list_resources()

    print("\n=== RESOURCES ===")
    for resource in resources_result.resources:
        print(f"\nName: {resource.name}")
        print(f"URI: {resource.uri}")
        print(f"Description: {resource.description}")

=== TOOLS ===

Name: data360_search_indicators
Description: Search for Data360 indicators with enriched metadata for selection.

Use when the user asks for data on a development topic (e.g. GDP, poverty, education).
Default to using the single `query` parameter for any single topic/indicator search. Use the `queries` or `query_groups` parameters ONLY when the request involves multiple topics or scopes (2 or more).
Provide exactly one of `query`, `queries`, or `query_groups`. One of these is strictly required.

### Parameter Selection Decision Tree (CRITICAL):
1. **Exactly 1 Topic** (e.g., "life expectancy" or "mortality rate") for any number of countries → you MUST use the single `query` parameter + `required_country`. Do NOT use `queries` with only one element, as it will fail. Do NOT combine multiple topics with 'and' or 'or' in `query` (e.g. do NOT use query="GDP and inflation").
2. **Multiple Topics, Same Country/Countries** (e.g., "life expectancy and GDP per capita" for Japan) → 

## Step 3: Load MCP Tools into LangChain
Convert MCP tools to LangChain tools for use in agents.

Do:

Connect to MCP server
Retrieve available tools
Convert to LangChain tool format (use langchain-mcp-adapters for automatic conversion)
Verify tools are accessible
Approach:

Use MultiServerMCPClient.get_langchain_tools() for automatic conversion (recommended)
Or manually wrap MCP tools as LangChain tools if needed
Verify tools are properly loaded and accessible

The difference from Step 2:
- session.list_tools() returns MCP tool definitions.
- client.get_tools() returns LangChain tool objects that an agent can use.

In [15]:
from langchain_core.tools import BaseTool

# Connect, retrieve MCP tools, and convert them to LangChain tools
tools = await client.get_tools()

# Verify that tools were returned
assert tools, "The server returned no tools."

# Verify their LangChain format
assert all(isinstance(tool, BaseTool) for tool in tools)

print(f"Loaded {len(tools)} LangChain tools.")

for tool in tools:
    print(f"\nName: {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Inputs: {tool.args}")

Loaded 16 LangChain tools.

Name: data360_search_indicators
Description: Search for Data360 indicators with enriched metadata for selection.

Use when the user asks for data on a development topic (e.g. GDP, poverty, education).
Default to using the single `query` parameter for any single topic/indicator search. Use the `queries` or `query_groups` parameters ONLY when the request involves multiple topics or scopes (2 or more).
Provide exactly one of `query`, `queries`, or `query_groups`. One of these is strictly required.

### Parameter Selection Decision Tree (CRITICAL):
1. **Exactly 1 Topic** (e.g., "life expectancy" or "mortality rate") for any number of countries → you MUST use the single `query` parameter + `required_country`. Do NOT use `queries` with only one element, as it will fail. Do NOT combine multiple topics with 'and' or 'or' in `query` (e.g. do NOT use query="GDP and inflation").
2. **Multiple Topics, Same Country/Countries** (e.g., "life expectancy and GDP per capita" 

### Step 4: Create Agent with MCP Tools
Build a LangChain agent that uses MCP tools.

Do:

Load MCP tools (from previous step)
Create a LangChain agent.
Test agent with MCP tools
Approach:

Initialize LLM (e.g., ChatOpenAI)
Create a prompt that describes available MCP capabilities
Pass MCP tools to the agent
Test with queries that require MCP tool usage

In [16]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

MODEL_NAME = "gpt-4o-mini"

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing from your environment.")

model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
)

 

In [17]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a research assistant using World Bank Data360. "
        "Use the available tools to answer data questions. "
        "Discover indicator identifiers through search; do not invent them. "
        "Report the indicator name, country, year, unit, and source "
        "when these are available. "
        "If data is missing or a tool fails, explain the limitation. "
        "Never invent numerical values."
    ),
)

In [ ]:
query = (
    "Find the latest available population total for Germany "
    "using World Bank Data360. Include the indicator, year, "
    "unit, and source."
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 30}, # limits how many execution steps the agent can take before it stops. It protects against endless loops.
)

print(result["messages"][-1].content)

The latest available population total for Germany is as follows:

- **Indicator**: Population, total
- **Year**: 2025
- **Unit**: Persons
- **Source**: World Development Indicators (WDI)

The total population is projected to be **83,491,249** persons.


In [19]:
from langchain_core.messages import AIMessage, ToolMessage

for message in result["messages"]:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print(f"\nTOOL REQUEST: {call['name']}")
            print("Arguments:", call["args"])

    elif isinstance(message, ToolMessage):
        print(f"\nTOOL RESPONSE: {message.name}")
        print(message.content)


TOOL REQUEST: data360_search_indicators
Arguments: {'query': 'population total', 'required_country': 'DEU'}

TOOL RESPONSE: data360_search_indicators
[{'type': 'text', 'text': '{"count":5,"total_count":200,"offset":0,"has_more":true,"next_offset":5,"indicators":[{"idno":"WB_WDI_SP_POP_TOTL","database_id":"WB_WDI","database_name":"World Development Indicators (WDI)","name":"Population, total","truncated_definition":"Total population is based on the de facto definition of population, which counts all residents regar","periodicity":"Annual","latest_data":"2025","time_period_range":"1960-2025","covers_country":{"DEU":true},"requested_country":"DEU","dimensions":null,"primary_source_of":null},{"idno":"WB_WDI_SP_POP_TOTL_FE_IN","database_id":"WB_WDI","database_name":"World Development Indicators (WDI)","name":"Population, female","truncated_definition":"Female population is based on the de facto definition of population, which counts all female residen","periodicity":"Annual","latest_data":

## Step 5: Access MCP Resources

In [24]:
async with client.session("data360") as session:
    resource_result = await session.read_resource(
        "data360://system-prompt"
    )
    runtime_result = await session.read_resource(
        "data360://context"
    )

# Extract the usage guidance
data360_guidance = "\n".join(
    content.text
    for content in resource_result.contents
    if hasattr(content, "text")
)

# Extract the current date and year
runtime_context = "\n".join(
    content.text
    for content in runtime_result.contents
    if hasattr(content, "text")
)

# Combine both resources
data360_context = (
    "Data360 usage guidance:\n" + data360_guidance
    + "\n\nRuntime context:\n" + runtime_context
)

print(data360_context)

Data360 usage guidance:
## Data360 Assistant

You are a tool-using assistant for World Bank Data360 indicators.

### Non-negotiable rule
If the user request requires indicator lookup, metadata, codes, or data values, you MUST call tools.
Do not answer with guesses. Do not stop after describing a plan.

### Operating loop (repeat until done)
1) If you need indicators or statistical series → call data360_search_indicators.
   - **CRITICAL: Search query is required**: You must always provide a search topic/term in `query`, `queries`, or `query_groups`. Do not omit it or pass empty values, even when filtering by database.
   - **CRITICAL: Parameter Selection Decision Tree**: Default to using the single `query` parameter for any single topic/indicator search. Use the `queries` or `query_groups` parameters ONLY when the request involves multiple topics or scopes (2 or more):
     - **Exactly 1 Topic** (e.g. "life expectancy") for any number of countries → you MUST use the single `query` para

In [22]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a research assistant using World Bank Data360. "
        "Use tools for factual data questions. "
        "Never invent indicator identifiers or numerical values. "
        "Report the source, indicator, country, year, and unit "
        "when available. Explain missing data or tool failures.\n\n"
        "Data360 usage guidance:\n"
        + data360_context
    ),
)

In [25]:
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 30},
)

print(result["messages"][-1].content)

The latest available population total for Germany is as follows:

- **Indicator**: Population, total
- **Year**: 2025
- **Value**: 83,491,249 persons
- **Source**: World Development Indicators (WDI)

This data reflects the total population based on the de facto definition, which counts all residents.


In [26]:
capabilities_result = await agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            "Erkläre mir auf Deutsch, was du mit dem verbundenen "
            "World Bank Data360 MCP-Server machen kannst. "
            "Orientiere dich an deinen verfügbaren Tools und der "
            "Data360-Anleitung. Nenne konkrete Tool-Namen und "
            "drei mögliche Beispielfragen. "
            "Führe noch keine Datenabfragen aus. "
            "Erfinde keine Fähigkeiten."
        )
    }]
})

print(capabilities_result["messages"][-1].content)

Ich kann auf den verbundenen World Bank Data360 MCP-Server zugreifen, um verschiedene Datenanfragen zu bearbeiten und Informationen zu Entwicklungsindikatoren bereitzustellen. Hier sind einige der spezifischen Tools, die ich verwenden kann, sowie Beispiele für Fragen, die ich beantworten kann:

1. **data360_search_indicators**: Mit diesem Tool kann ich nach spezifischen Indikatoren suchen, die mit Entwicklungsfragen wie Wirtschaft, Gesundheit oder Bildung zusammenhängen. 
   - Beispiel Frage: „Was ist das BIP pro Kopf in Deutschland?“

2. **data360_get_data**: Dieses Tool ermöglicht es mir, tatsächliche Datenwerte für bestimmte Indikatoren und Länder abzurufen.
   - Beispiel Frage: „Wie hoch war die Arbeitslosenquote in Spanien im Jahr 2022?“

3. **data360_rank_countries**: Mit diesem Tool kann ich Länder nach bestimmten Indikatoren für ein bestimmtes Jahr rangieren.
   - Beispiel Frage: „Welche zehn Länder haben die höchste Lebenserwartung im Jahr 2021?“

Ich kann auch andere Funktion

## Step 6: Build a Complete MCP-Enabled Agent


In [27]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a research assistant using World Bank Data360. "
        "Use the available tools for factual data questions. "
        "Search for indicator identifiers rather than guessing them. "
        "Check indicator definitions, units, and country coverage. "
        "Support numerical claims with returned tool data. "
        "Report the indicator, country, observation year, unit, and source. "
        "Distinguish the current year from the latest available data year. "
        "For comparisons, use the same indicator, unit, and observation year. "
        "Explain missing data and tool failures. Never invent values.\n\n"
        "Data360 usage guidance:\n"
        + data360_guidance
        + "\n\nRuntime context:\n"
        + runtime_context
    ),
)

In [31]:
query = (
    " wie hat sich das pro kips bip in deutschland erändert seit 2010. schlüssel das auf nach geschlacht, bundesland. wie könnte man das noch weiter aufschlüsslen mit der verfügbaren daten? erklär auch die limitationen" \
    ""
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 30},
)

print(result["messages"][-1].content)

### Entwicklung des BIP pro Kopf in Deutschland seit 2010

Hier sind die Daten zum BIP pro Kopf in Deutschland (in konstanten US-Dollar von 2015) von 2010 bis 2024:

| Jahr | BIP pro Kopf (USD, konstant 2015) |
|------|-----------------------------------|
| 2010 | 38,526.74                         |
| 2011 | 40,722.48                         |
| 2012 | 40,834.51                         |
| 2013 | 40,884.89                         |
| 2014 | 41,602.47                         |
| 2015 | 41,929.75                         |
| 2016 | 42,516.93                         |
| 2017 | 43,543.48                         |
| 2018 | 43,905.86                         |
| 2019 | 44,235.27                         |
| 2020 | 42,372.87                         |
| 2021 | 44,011.02                         |
| 2022 | 44,817.13                         |
| 2023 | 44,368.99                         |
| 2024 | 44,027.76                         |

### Aufschlüsselung nach Geschlecht und Bundesland

Die Daten können

In [40]:
query = (
    " ich will schauen wo in deutschland gearbeitet wird, in welchen sektoren udn wie sich das über die jahre verändert hat" \
    ""
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 30},
)

print(result["messages"][-1].content)

Hier sind einige relevante Indikatoren zur Beschäftigung in Deutschland, die die Sektoren und deren Veränderungen über die Jahre darstellen:

1. **Indikator: Employment by sector (%)**
   - **ID:** WB_GS_SL_EMPL_ZS
   - **Datenbank:** Gender Statistics
   - **Definition:** Beschäftigung wird definiert als Personen im erwerbsfähigen Alter, die in irgendeiner Aktivität zur Produktion von Gütern oder zur Bereitstellung von Dienstleistungen gegen Bezahlung oder Gewinn tätig waren.
   - **Einheit:** Prozent
   - **Zeitraum:** 1960-2023
   - **Letzte Daten:** 2023
   - **Link zur CSV:** [CSV Download](https://data360files.worldbank.org/data360-data/data/WB_GS/WB_GS_SL_EMPL_ZS.csv)

2. **Indikator: Number of employed employees, by industry**
   - **ID:** WB_WWBI_BI_EMP_TOTL_NO
   - **Datenbank:** Worldwide Bureaucracy Indicators (WWBI)
   - **Definition:** Anzahl der beschäftigten Mitarbeiter nach Branche.
   - **Einheit:** Personen
   - **Zeitraum:** 2000-2022
   - **Letzte Daten:** 2022
   

In [36]:
from langchain_core.messages import AIMessage, ToolMessage

for message in result["messages"]:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print(f"\nREQUEST: {call['name']}")
            print("Arguments:", call["args"])

    elif isinstance(message, ToolMessage):
        print(f"\nRESPONSE: {message.name}")
        print(message.content)


REQUEST: data360_search_indicators
Arguments: {'query': 'GDP per capita', 'required_country': 'DEU'}

RESPONSE: data360_search_indicators
[{'type': 'text', 'text': '{"count":5,"total_count":200,"offset":0,"has_more":true,"next_offset":5,"indicators":[{"idno":"WB_WDI_NY_GDP_PCAP_CD","database_id":"WB_WDI","database_name":"World Development Indicators (WDI)","name":"GDP per capita (current US$)","truncated_definition":"Gross domestic product is the total income earned through the production of goods and services in an","periodicity":"Annual","latest_data":"2024","time_period_range":"1960-2024","covers_country":{"DEU":true},"requested_country":"DEU","dimensions":null,"primary_source_of":null},{"idno":"WB_WDI_NY_GDP_PCAP_KD","database_id":"WB_WDI","database_name":"World Development Indicators (WDI)","name":"GDP per capita (constant 2015 US$)","truncated_definition":"Gross domestic product is the total income earned through the production of goods and services in an","periodicity":"Annual"